# PCG-to-ECG Signal Reconstruction: Evaluation Metrics

**Ablation Study Code** for: *PCG-to-ECG Signal Reconstruction Using Conditional GAN with Multi-Scale Attention* (EUSIPCO 2026)

**Authors:** Zafar Ahmad, Nazia Perwaiz, Muhammad Salman Khan

---

**Signal-level metrics:** PSNR, SNR, SSIM, Pearson correlation, MAE, RMSE

**Clinical metrics:** R-peak amplitude error, QRS width error

The paper (Section IV-C) shows that signal-level and clinical metrics can disagree: a model may score SSIM = 0.912 yet have 9.2x worse R-peak error, because PSNR/SSIM are dominated by the ~70% low-amplitude baseline.

## Imports

In [ ]:
import numpy as np
import torch
from scipy.stats import pearsonr
from skimage.metrics import structural_similarity as compare_ssim
import scipy.signal as signal

---
## R-Peak Detection

In [ ]:
def detect_r_peaks(ecg, fs=8000, distance=400, prominence=0.5):
    """
    Detect R-peaks with scipy.signal.find_peaks.

    Parameters
    ----------
    ecg : np.ndarray, shape (T,)
    fs  : int, sampling rate (default 8 kHz)
    distance : int, minimum inter-peak distance in samples
               (400 = 50 ms at 8 kHz)
    prominence : float, minimum peak prominence

    Returns
    -------
    peaks : np.ndarray of peak indices
    props : dict of peak properties
    """
    return signal.find_peaks(ecg, distance=distance,
                             prominence=prominence)

---
## Signal-Level Metrics

In [ ]:
def calc_psnr(real, fake):
    """
    Peak Signal-to-Noise Ratio (dB).

    PSNR = 20 * log10( max|real| / sqrt(MSE) )

    Because signals are z-score normalized (range ~ 6 units),
    raw PSNR values are NOT comparable to studies that normalize
    to [0, 1] (range = 1 unit). See paper Limitation 4.
    """
    if torch.is_tensor(real):
        mse = torch.mean((real - fake) ** 2)
        if mse < 1e-10:
            return 100.0
        peak = torch.max(torch.abs(real))
        return (20 * torch.log10(peak / torch.sqrt(mse))).item()
    else:
        mse = np.mean((real - fake) ** 2)
        if mse < 1e-10:
            return 100.0
        peak = np.max(np.abs(real))
        return float(20 * np.log10(peak / np.sqrt(mse)))

In [ ]:
def calc_snr(real, fake):
    """
    Signal-to-Noise Ratio (dB).
    SNR = 10 * log10( sum(real^2) / sum((real - fake)^2) )
    """
    if torch.is_tensor(real):
        real, fake = real.cpu().numpy(), fake.cpu().numpy()
    real, fake = real.flatten(), fake.flatten()
    noise = real - fake
    sig_power = np.sum(real ** 2)
    noise_power = np.sum(noise ** 2) + 1e-12
    return float(10 * np.log10(sig_power / noise_power))

In [ ]:
def calc_correlation(real, fake):
    """Pearson correlation coefficient (per-sample, then averaged)."""
    if torch.is_tensor(real):
        real, fake = real.cpu().numpy(), fake.cpu().numpy()
    if real.ndim == 3:
        real = real.reshape(real.shape[0], -1)
        fake = fake.reshape(fake.shape[0], -1)
    if real.ndim == 1:
        real, fake = real[np.newaxis], fake[np.newaxis]
    corrs = []
    for i in range(len(real)):
        try:
            c, _ = pearsonr(real[i], fake[i])
            corrs.append(c)
        except Exception:
            pass
    return float(np.mean(corrs)) if corrs else 0.0

In [ ]:
def calc_ssim(real, fake):
    """
    Structural Similarity Index (SSIM).
    Both signals are mapped to [0, 1] before comparison.
    """
    if torch.is_tensor(real):
        real, fake = real.cpu().numpy(), fake.cpu().numpy()
    if real.ndim == 3:
        real, fake = real.squeeze(1), fake.squeeze(1)
    if real.ndim == 1:
        real, fake = real[np.newaxis], fake[np.newaxis]
    vals = []
    for i in range(len(real)):
        r = real[i]
        f = fake[i]
        rng = r.max() - r.min() + 1e-8
        r_n = (r - r.min()) / rng
        f_n = (f - f.min()) / (f.max() - f.min() + 1e-8)
        vals.append(compare_ssim(r_n, f_n, data_range=1.0))
    return float(np.mean(vals)) if vals else 0.0

In [ ]:
def calc_mae(real, fake):
    """Mean Absolute Error."""
    if torch.is_tensor(real):
        return torch.mean(torch.abs(real - fake)).item()
    return float(np.mean(np.abs(real - fake)))


def calc_rmse(real, fake):
    """Root Mean Squared Error."""
    if torch.is_tensor(real):
        return torch.sqrt(torch.mean((real - fake) ** 2)).item()
    return float(np.sqrt(np.mean((real - fake) ** 2)))

---
## Clinical Metrics

These capture clinically important waveform features that PSNR/SSIM miss:

- **R-peak amplitude error:** Mean absolute difference between ground truth and reconstructed R-peak amplitudes. R-peak amplitude signals ventricular hypertrophy.
- **QRS width error:** Mean absolute difference in QRS complex duration (ms). QRS > 120 ms signals bundle-branch block.

In [ ]:
def calc_r_peak_error(real_ecg, fake_ecg, fs=8000):
    """
    R-peak amplitude error: E_R = (1/K) * sum |y(t_k) - y_hat(t_k)|

    R-peaks are detected on the ground truth signal. The
    reconstructed signal is evaluated at the same time indices.
    """
    if torch.is_tensor(real_ecg):
        real_ecg = real_ecg.cpu().numpy()
    if torch.is_tensor(fake_ecg):
        fake_ecg = fake_ecg.cpu().numpy()
    if real_ecg.ndim == 3:
        real_ecg, fake_ecg = real_ecg.squeeze(1), fake_ecg.squeeze(1)
    if real_ecg.ndim == 1:
        real_ecg = real_ecg[np.newaxis]
        fake_ecg = fake_ecg[np.newaxis]

    errors = []
    for i in range(len(real_ecg)):
        peaks, _ = detect_r_peaks(real_ecg[i], fs=fs)
        if len(peaks) == 0:
            continue
        err = np.mean(np.abs(real_ecg[i][peaks] - fake_ecg[i][peaks]))
        errors.append(err)
    return float(np.mean(errors)) if errors else 0.0

In [ ]:
def _qrs_width_at_peak(ecg, peak_idx, fs=8000, threshold=0.5):
    """
    Estimate QRS width (ms) around a single R-peak by finding
    the points where amplitude falls below threshold * |R-peak|.
    """
    amp = abs(ecg[peak_idx])
    thr = threshold * amp

    # search backward for Q onset
    q = peak_idx
    for j in range(peak_idx - 1,
                   max(peak_idx - 200, 0), -1):
        if abs(ecg[j]) < thr:
            q = j
            break

    # search forward for S offset
    s = peak_idx
    for j in range(peak_idx + 1,
                   min(peak_idx + 200, len(ecg))):
        if abs(ecg[j]) < thr:
            s = j
            break

    return (s - q) / fs * 1000.0   # ms


def calc_qrs_width_error(real_ecg, fake_ecg, fs=8000):
    """
    QRS width error (ms): E_QRS = (1/K) * sum |w_true - w_pred|
    Normal QRS width is 80-120 ms.
    """
    if torch.is_tensor(real_ecg):
        real_ecg = real_ecg.cpu().numpy()
    if torch.is_tensor(fake_ecg):
        fake_ecg = fake_ecg.cpu().numpy()
    if real_ecg.ndim == 3:
        real_ecg, fake_ecg = real_ecg.squeeze(1), fake_ecg.squeeze(1)
    if real_ecg.ndim == 1:
        real_ecg = real_ecg[np.newaxis]
        fake_ecg = fake_ecg[np.newaxis]

    errors = []
    for i in range(len(real_ecg)):
        peaks, _ = detect_r_peaks(real_ecg[i], fs=fs)
        for pk in peaks:
            w_real = _qrs_width_at_peak(real_ecg[i], pk, fs)
            w_fake = _qrs_width_at_peak(fake_ecg[i], pk, fs)
            errors.append(abs(w_real - w_fake))
    return float(np.mean(errors)) if errors else 0.0

---
## Aggregate: All Metrics at Once

In [ ]:
def calc_all_metrics(real, fake, include_clinical=True, fs=8000):
    """
    Return a dict of all metrics for a batch of signals.

    Parameters
    ----------
    real, fake : Tensor [B, 1, T] or np.ndarray
    include_clinical : bool, if True compute R-peak and QRS
                       metrics (slower, skip during training)
    fs : int, sampling rate

    Returns
    -------
    dict[str, float]
    """
    m = {
        'psnr':        calc_psnr(real, fake),
        'snr':         calc_snr(real, fake),
        'ssim':        calc_ssim(real, fake),
        'correlation': calc_correlation(real, fake),
        'mae':         calc_mae(real, fake),
        'rmse':        calc_rmse(real, fake),
    }
    if include_clinical:
        m['r_peak_error']    = calc_r_peak_error(real, fake, fs)
        m['qrs_width_error'] = calc_qrs_width_error(real, fake, fs)
    return m

# Quick test with random signals
real = np.random.randn(2, 64000).astype(np.float32)
fake = real + 0.1 * np.random.randn(2, 64000).astype(np.float32)
metrics = calc_all_metrics(real, fake, include_clinical=True)
for k, v in metrics.items():
    print(f"  {k:20s} = {v:.4f}")